# Pipeline smoke test

This notebook wires the pieces together on a few images:

`SpectralWasteDataset` → `DataLoader` → a **placeholder** model → cross-entropy with hyperspectral class weights → foreground mIoU.

**The model below is a placeholder.** It is a single 1×1 convolution so the loop has something to run. The CNN control and the band-token transformer are the group's own work packages. Build those yourselves. This notebook does not implement them.


## Colab setup

Skip this cell on your own machine. In Colab it clones the repo, installs the libraries, and downloads the data (about 23 GB).


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# This cell is only for Google Colab. On your own machine it does nothing.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content")
    repo = Path("/content/255-final-project")
    if not repo.is_dir():
        subprocess.check_call(
            ["git", "clone", "https://github.com/scottn66/255-final-project.git", str(repo)]
        )
    os.chdir(repo)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )
    if not os.environ.get("SW_ROOT"):
        subprocess.check_call(["bash", "scripts/get_data.sh", "/content/data"])
        os.environ["SW_ROOT"] = "/content/data/spectralwaste_segmentation"


## Find the data

Same rule as the first notebook: `SW_ROOT` points at the folder that contains `hyper/` and `labels_hyper_lt/`.


In [2]:
import os

if not os.environ.get("SW_ROOT"):
    raise RuntimeError(
        "SW_ROOT is not set, so this notebook cannot find the dataset.\n\n"
        "From the repo root, download it (about 23 GB) and export the path "
        "the script prints:\n\n"
        "    bash scripts/get_data.sh\n"
        "    export SW_ROOT=\"/absolute/path/printed/by/the/script\"\n\n"
        "That folder contains hyper/ and labels_hyper_lt/.\n"
        "Restart the kernel, then run this notebook from the top."
    )
print("SW_ROOT =", os.environ["SW_ROOT"])


SW_ROOT = ./data/spectralwaste_segmentation


## Dataset and loader

Training uses a left-right flip. We only take a handful of images, because this is a wiring check, not a training run.
Validation is unflipped.


In [3]:
import sys
from pathlib import Path


def _add_src():
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        src = cand / "src"
        if (src / "sw" / "__init__.py").is_file():
            if str(src) not in sys.path:
                sys.path.insert(0, str(src))
            return
    raise RuntimeError(
        "Could not import sw. Start Jupyter from the repo root or from the notebooks/ folder."
    )


_add_src()

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from sw import (
    CLASS_NAMES,
    HYPER_LT_TRAIN_PIXEL_COUNTS,
    ConfusionMeter,
    SpectralWasteDataset,
    class_weights,
)

train_ds = SpectralWasteDataset(split="train", random_flip=True)
val_ds = SpectralWasteDataset(split="val")
sample, _mask = train_ds[0]
in_channels = sample.shape[0]

n_train = min(4, len(train_ds))
loader = DataLoader(
    Subset(train_ds, range(n_train)),
    batch_size=2,
    shuffle=True,
)
print(f"using {n_train} of {len(train_ds)} train images, {in_channels} bands, val images available: {len(val_ds)}")


using 4 of 514 train images, 224 bands, val images available: 167


## Placeholder model

One 1×1 convolution maps each pixel's spectrum to 7 class scores. It does not look at neighbouring pixels, and it does not use attention.

Replace this later with your own model. Keep the input as `(batch, bands, 256, 256)` and the output as `(batch, 7, 256, 256)` so the loss and the metric below stay the same.


In [4]:
class Placeholder1x1(nn.Module):
    """Stand-in head: one 1x1 conv. Not the CNN control, and not the band-token transformer."""

    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


model = Placeholder1x1(in_channels, num_classes=len(CLASS_NAMES))
print(model)


Placeholder1x1(
  (conv): Conv2d(224, 7, kernel_size=(1, 1), stride=(1, 1))
)


## A few training steps

`class_weights()` turns the `labels_hyper_lt` train pixel counts into inverse-frequency weights (mean 1).
Filament has very few pixels, so its weight is large. These counts are not the RGB counts in `meta.json`.
The inverse-frequency weights span about 900× (background ~0.006, filament ~5.4), which is why a 5-step placeholder never predicts background. The team can choose a different weighting later.


In [5]:
weights = class_weights()
print("class weights for the hyperspectral masks:")
for name, n, w in zip(CLASS_NAMES, HYPER_LT_TRAIN_PIXEL_COUNTS, weights.tolist()):
    print(f"  {name:12} pixels={n:10d}  weight={w:8.3f}")

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
step = 0
while step < 5:
    for cubes, masks in loader:
        loss = criterion(model(cubes), masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        step += 1
        print(f"step {step}  loss {loss.item():.4f}")
        if step >= 5:
            break


class weights for the hyperspectral masks:
  background   pixels=  27612233  weight=   0.006
  film         pixels=   1819248  weight=   0.098
  basket       pixels=   1060150  weight=   0.169
  cardboard    pixels=    220623  weight=   0.810
  video_tape   pixels=    397471  weight=   0.449
  filament     pixels=     33101  weight=   5.397
  bag          pixels=   2542678  weight=   0.070


step 1  loss 1.9095


step 2  loss 1.7930


step 3  loss 1.7224
step 4  loss 1.6535
step 5  loss 1.5845


## Foreground mIoU on the validation split

The score is the mean IoU of classes 1–6. Background (class 0) is left out of that mean.
The per-class row still prints background so you can see it.
A nan means the class is absent from both target and prediction and is excluded from the mean.

A placeholder trained for five steps will score poorly. The published hyperspectral numbers (MiniNet-v2 52.8, SegFormer-B0 54.3) are context for later, when the real models exist.


In [6]:
import math

meter = ConfusionMeter()
model.eval()
with torch.no_grad():
    for index in range(len(val_ds)):
        cube, mask = val_ds[index]
        logits = model(cube.unsqueeze(0))
        meter.update(logits, mask.unsqueeze(0))

per_class = meter.per_class_iou()
n_fg = sum(not math.isnan(float(value)) for value in per_class[1:])
print(f"foreground mIoU on all {len(val_ds)} val images: {meter.fg_miou():.4f}")
print(f"averaged {n_fg} of 6 foreground classes")
print("per-class IoU:")
for name, iou in zip(CLASS_NAMES, per_class):
    print(f"  {name:12} {iou:.4f}")


foreground mIoU on all 167 val images: 0.0126
averaged 6 of 6 foreground classes
per-class IoU:
  background   0.0000
  film         0.0000
  basket       0.0000
  cardboard    0.0000
  video_tape   0.0000
  filament     0.0000
  bag          0.0758


## Where the real models go

This file showed the path from disk to a loss and a metric. The two models the group will write are:

* a small CNN control
* a band-token transformer (the attention blocks in `sw.attention` and the wavelength encoding in `sw.pe` are available if you want them)

Put those in your own modules. Point them at `SpectralWasteDataset`, `class_weights()`, and `ConfusionMeter`.
